In [1]:
print("all good")

all good


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

In [3]:
BASE_DIR =Path().resolve()

In [5]:
images_root = BASE_DIR/"images"

In [6]:
images_root

WindowsPath('C:/Users/pkdee/projects/semantic-image-search/notebook/images')

In [7]:
Model_id = "ViT-B-32__laion2b-s34b-b79k"

In [8]:
Model_id = "openai/clip-vit-base-patch32"

In [9]:
load_dotenv()

True

In [10]:
from langchain_experimental.open_clip import OpenCLIPEmbeddings

embedder = OpenCLIPEmbeddings(
    model_name="ViT-B-32",
    checkpoint="openai", 
    device="cpu"
)

c:\Users\pkdee\projects\semantic-image-search\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\pkdee\projects\semantic-image-search\.venv\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


In [11]:
dog =str(images_root/"animal"/"dog.avif")

In [12]:
dog

'C:\\Users\\pkdee\\projects\\semantic-image-search\\notebook\\images\\animal\\dog.avif'

In [16]:
embedder.embed_image([dog])

[[0.009442105889320374,
  0.0043057664297521114,
  0.015094812028110027,
  -0.022211700677871704,
  0.009254470467567444,
  -0.08072568476200104,
  0.048336803913116455,
  0.013864682987332344,
  -0.02956193871796131,
  0.014887229539453983,
  -0.02786475233733654,
  -0.011823220178484917,
  0.026262788102030754,
  0.03963570296764374,
  0.08643617480993271,
  0.005739129148423672,
  0.012127313762903214,
  -0.023506172001361847,
  0.028832286596298218,
  -0.0031389202922582626,
  -0.046615101397037506,
  0.004815736785531044,
  0.01712637208402157,
  -0.015088320709764957,
  -0.019008953124284744,
  -0.009711461141705513,
  0.020047245547175407,
  -0.006599773187190294,
  -0.005444344133138657,
  0.0008791219443082809,
  0.0028756330721080303,
  0.02312873862683773,
  -0.05271982401609421,
  -0.022468527778983116,
  0.045499883592128754,
  0.048292376101017,
  0.019350146874785423,
  0.031888943165540695,
  -0.031895097345113754,
  0.11744248867034912,
  -0.04508133977651596,
  0.0056

In [13]:
url = os.getenv("API_ENDPOINT")

In [14]:
url

'https://d7b5ea27-f09f-4a75-bf6c-75c70c688518.us-east-1-1.aws.cloud.qdrant.io:6333'

In [15]:
api_key = os.getenv("QDRANT_API_KEY")

In [16]:
from qdrant_client import QdrantClient

In [18]:
qdrant_client = QdrantClient(url = url , api_key = api_key)

In [21]:
collections  = qdrant_client.get_collections()

In [27]:
collections

CollectionsResponse(collections=[])

In [24]:
collection_name = "semantic-image-search"
vector_size = 512

In [25]:
from qdrant_client.http import models

In [26]:
qdrant_client.create_collection(
    collection_name=collection_name,
    vectors_config = models.VectorParams(size=vector_size,
                                         distance=models.Distance.COSINE,))

True

In [30]:
collections = qdrant_client.get_collections().collections

In [31]:
existing_names = {c.name for c in collections}

In [32]:
existing_names

{'semantic-image-search'}

In [33]:
if collection_name not in existing_names:
    print(f"creating collection:{collection_name}")
    qdrant_client.create_collection(
        collection_name=collection_name,
        vectors_config = models.VectorParams(size=vector_size,
                                            distance=models.Distance.COSINE,))
else:
    print(f"collection {collection_name} already exists")

collection semantic-image-search already exists


In [34]:
import os
import numpy as np
from PIL import Image
from uuid import uuid4

In [ ]:
def index_image(image_path, category=None):
    img_embedding = embedder.embed_image([image_path])[0]
    emb = np.array(img_embedding).tolist()

    payload = {
        "filename": os.path.basename(image_path),
        "path": image_path,
        "category": category,
    }
    point = models.PointStruct(
        id=str(uuid4()),
        vector=emb,
        payload=payload,
    )
    qdrant_client.upsert(
        collection_name=collection_name,
        points=[point],
    )

    print(f"Indexed image: {image_path}")
    